# CoSS-DF — Colab Runner

This notebook implements the persistent workflow

**Google Drive → `/content` SSD → GPU/CPU computation → checkpoints/results back to Drive.**

After a Colab disconnect, rerun this notebook and then rerun the same pipeline command. Completed Drive-backed chunks, candidates, and outer folds are reused automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 1) Repository, paths, and active dataset
from pathlib import Path
import os, subprocess

REPO_URL = "GITHUB_URL" #@param {type:"string"}
REPO_DIR = Path("/content/coss-df")
DRIVE_ROOT = Path("/content/drive/MyDrive/CoSS_DF") #@param {type:"string"}
DATASET = "kth_tips2b" #@param ["kth_tips2b", "fmd", "kylberg"]

os.environ["COSSDF_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["COSSDF_LOCAL_ROOT"] = "/content/cossdf_work"

if not REPO_DIR.exists():
    if REPO_URL == "GITHUB_URL":
        raise ValueError("Replace GITHUB_URL with the repository URL before running this cell.")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

PROJECT = REPO_DIR / "software" / "pipeline_source"
print("Project:", PROJECT)
print("Drive root:", DRIVE_ROOT)
print("Active dataset:", DATASET)


In [ ]:
#@title 2) Install the browsable source tree
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-colab.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", str(PROJECT)])
print("Installed project:", PROJECT)


In [ ]:
#@title 3) Validate package before touching paper data
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pytest", "-q", str(PROJECT / "tests")])


## Recommended stage-by-stage run

If you expect a short Colab session, run the cells below one stage at a time. Every persistent output is written to Drive.


In [ ]:
#@title 4A) Download once to Drive + extract persistently
import subprocess
subprocess.check_call(["cossdf", "prepare", "--dataset", DATASET])


In [ ]:
#@title 4B) Copy active dataset Drive → /content SSD + audit + frozen folds
import subprocess
subprocess.check_call(["cossdf", "audit", "--dataset", DATASET])


In [ ]:
#@title 4C) Chunked ResNet18/DCT/LBP feature extraction
import subprocess
subprocess.check_call(["cossdf", "features", "--dataset", DATASET])


In [ ]:
#@title 4D) Nested selection + outer-fold paper models
import subprocess
subprocess.check_call(["cossdf", "run", "--dataset", DATASET])


In [ ]:
#@title 4E) Bootstrap, reliability and theorem-verification outputs
import subprocess
subprocess.check_call(["cossdf", "analyze", "--dataset", DATASET])


## One-command mode

This is safe to rerun after a disconnect. It checks/reuses persistent Drive artifacts and restages only the active dataset to the fresh `/content` SSD.


In [ ]:
#@title 5) Run/resume complete main pipeline
import subprocess
subprocess.check_call(["cossdf", "all", "--dataset", DATASET])


## Supplementary analyses

These are prespecified but are best run after the main paper pipeline has completed.


In [ ]:
#@title 6A) Block-grid sensitivity
import subprocess
subprocess.check_call(["cossdf", "grid", "--dataset", DATASET])


In [ ]:
#@title 6B) eta/gamma sensitivity (KTH-TIPS2b only)
import subprocess
if DATASET == "kth_tips2b":
    subprocess.check_call(["cossdf", "sensitivity", "--dataset", DATASET])
else:
    print("Skipped: prespecified only for KTH-TIPS2b.")


In [ ]:
#@title 7) Inspect persistent outputs
from pathlib import Path
result_dir = DRIVE_ROOT / "results" / DATASET / "paper_v1_0"
print("Results:", result_dir)
if result_dir.exists():
    for p in sorted(result_dir.rglob("*")):
        if p.is_file():
            print(p.relative_to(result_dir), f"{p.stat().st_size/1024:.1f} KB")


### Resume rule

If the runtime disconnects:

1. reconnect to a GPU runtime;
2. mount Drive again;
3. rerun cells 1–3 to restore the pipeline code under `/content`;
4. rerun cell 5 (or the interrupted stage cell).

Do **not** delete the Drive `checkpoints/` directory. The local `/content` dataset will be copied again automatically because the Colab SSD is ephemeral.
